import torch → PyTorch library import karta hai.
import torch.nn as nn → Neural Network ke layers/models banane ke liye.
import torch.optim as optim → Optimizer (Adam, SGD etc.) ke liye.
from torch.utils.data import DataLoader → Data ko batches mein load karne ke liye.
from torchvision import datasets, transforms → Image datasets aur image preprocessing ke liye.
datasets → Ready-made datasets (MNIST, CIFAR etc.) ke liye.
transforms → Images ko resize, normalize, tensor mein convert etc. karne ke liye.


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.cuda.is_available() → Check karta hai GPU (CUDA) available hai ya nahi.
"cuda" if ... else "cpu" → GPU hai to cuda, warna cpu choose karta hai.
torch.device(...) → Selected device ko set karta hai.
print(f"Using device: {device}") → Bataata hai ki code GPU par chalega ya CPU par.

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device : {device}")

Using device : cuda


transforms.Compose([...]) → Multiple transformations ko ek pipeline mein combine karta hai.
transforms.ToTensor() → Image ko PyTorch Tensor mein convert karta hai aur pixel values ko generally 0–1 range mein laata hai.
transforms.Normalize((0.1307,), (0.3801,)) → MNIST images ko normalize karta hai using:
0.1307 → mean
0.3801 → standard deviation

In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3801,))  #MNIST DATASET MEAN AND STANDARD-DEVIATION
])

datasets.MNIST(...) → MNIST handwritten digits dataset load karta hai.
Training dataset
root="./data" → Dataset ./data folder mein save hoga.
train=True → Training data lega.
download=True → Dataset nahi mila to automatically download karega.
transform=transform → Jo transformations humne banaye (ToTensor + Normalize) woh apply karega.
Test dataset
train=False → Testing data lega.
Baaki parameters same hain.

👉 Short mein:
train_dataset → model ko sikhane ke liye
test_dataset → model ki performance check karne ke liye.

In [6]:
train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

DataLoader(train_dataset, ...) → Training dataset ko batches mein load karta hai.
batch_size=64 → Ek baar mein 64 images model ko dega.
shuffle=True → Training data ko har epoch randomly shuffle karega.
DataLoader(test_dataset, ...) → Test dataset ko batches mein load karta hai.
batch_size=64 → Ek batch mein 64 test images.
shuffle=False → Test data ko shuffle nahi karega.

👉 Short mein: DataLoader dataset ko small batches mein model tak pahunchata hai.

In [7]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

1. CNN Class

class CNN(nn.Module):
→ CNN model ki class banata hai.

def __init__(self):
→ Model ke layers define/initialize karta hai.

super().__init__()
→ nn.Module ko initialize karta hai.

2. Convolution Layers

self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
→ 1-channel image ko 16 feature maps mein convert karta hai using 3×3 filter.

self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
→ 16 feature maps ko 32 feature maps mein convert karta hai.

3. Pooling Layer

self.pool = nn.MaxPool2d(2, 2)
→ Feature map ka height aur width half karta hai.

4. Fully Connected Layers

self.fc1 = nn.Linear(32 * 7 * 7, 128)
→ Flattened features ko 128 neurons mein convert karta hai.

self.fc2 = nn.Linear(128, 10)
→ 128 neurons se 10 outputs deta hai, yani digits 0–9.

5. Activation Function

self.relu = nn.ReLU()
→ Non-linearity add karta hai aur negative values ko remove karta hai.

⚠️ Tumhare original code mein nn.Relu() tha. Correct spelling nn.ReLU() hai.

Forward Pass

def forward(self, x):
→ Define karta hai ki input model ke andar kis sequence mein process hoga.

x = self.pool(self.relu(self.conv1(x)))
→ Conv1 → ReLU → MaxPool apply karta hai.

x = self.pool(self.relu(self.conv2(x)))
→ Conv2 → ReLU → MaxPool apply karta hai.

x = x.view(x.size(0), -1)
→ Feature maps ko 1D vector (Flatten) mein convert karta hai.

x = self.relu(self.fc1(x))
→ Fully connected layer + ReLU apply karta hai.

x = self.fc2(x)
→ Final 10 class scores generate karta hai.

return x
→ Model ka final output return karta hai.

6. Model Creation

model = CNN().to(device)
→ CNN model create karta hai aur GPU/CPU par bhejta hai.

Overall Flow

28×28 Image → Conv1 → ReLU → Pool → Conv2 → ReLU → Pool → Flatten → FC1 → ReLU → FC2 → 10 Outputs

👉 Main purpose: Handwritten MNIST digit ko 0–9 mein classify karna.

In [8]:
class CNN(nn.Module) :
  def __init__(self) :
    super().__init__()
    self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
    self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
    self.pool = nn.MaxPool2d(2, 2)
    self.fc1 = nn.Linear(32 * 7 * 7, 128)
    self.fc2 = nn.Linear(128, 10)
    self.relu = nn.ReLU()

  def forward(self, x) :
    x = self.pool(self.relu(self.conv1(x)))
    x = self.pool(self.relu(self.conv2(x)))
    x = x.view(x.size(0), -1)
    x = self.relu(self.fc1(x))
    x = self.fc2(x)
    return x

model = CNN().to(device)

Loss Function & Optimizer

criterion = nn.CrossEntropyLoss()
→ Model ki prediction aur actual label ke beech error (loss) calculate karta hai.

optimizer = optim.Adam(model.parameters(), lr=0.00001)
→ Model ke weights update karta hai taaki loss kam ho.

model.parameters() → Model ke saare learnable weights.
lr=0.00001 → Learning rate, yani weights kitni speed se update honge.


In [9]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.00001)

num_epochs = 5
→ Model ko 5 baar poore training dataset par train karega.

for epoch in range(num_epochs):
→ 5 epochs ke liye training loop chalata hai.

model.train()
→ Model ko training mode mein set karta hai.

total_loss = 0
→ Total loss ko 0 se start karta hai.

correct = 0
→ Correct predictions count karne ke liye 0 se start karta hai.

Batch Training

for images, labels in train_loader:
→ Training data ko batch-by-batch leta hai.

images, labels = images.to(device), labels.to(device)
→ Images aur labels ko GPU/CPU par bhejta hai.

optimizer.zero_grad()
→ Previous batch ke gradients clear karta hai.

outputs = model(images)
→ Images ko model mein pass karke predictions nikalta hai.

loss = criterion(outputs, labels)
→ Predictions aur actual labels ke beech error calculate karta hai.

loss.backward()
→ Loss ke basis par gradients calculate karta hai.

optimizer.step()
→ Gradients use karke model ke weights update karta hai.

Loss & Accuracy

total_loss += loss.item()
→ Current batch ka loss total loss mein add karta hai.

correct += (outputs.argmax(1) == labels).sum().item()
→ Kitni predictions correct hain, unko count karta hai.

train_acc = correct / len(train_dataset)
→ Total correct predictions se training accuracy calculate karta hai.

print(...)
→ Har epoch ka Loss aur Accuracy display karta hai.

Overall Flow

Data → Prediction → Loss → Backpropagation → Weights Update → Accuracy

👉 Ye code actually CNN ko train karta hai.

In [11]:
num_epochs = 5

for epoch in range(num_epochs) :
  model.train()
  total_loss = 0
  correct = 0

  for images, labels in train_loader :
    images, labels = images.to(device), labels.to(device)

    optimizer.zero_grad()
    outputs = model(images)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

    total_loss += loss.item()
    correct += (outputs.argmax(1) == labels).sum().item()

  train_acc = correct / len(train_dataset)
  print(f"Epoch {epoch+1}/{num_epochs} | Loss: {total_loss/len(train_loader):.4f} | Train Acc: {train_acc:.4f}")

Epoch 1/5 | Loss: 1.8300 | Train Acc: 0.6417
Epoch 2/5 | Loss: 0.7744 | Train Acc: 0.8441
Epoch 3/5 | Loss: 0.4502 | Train Acc: 0.8872
Epoch 4/5 | Loss: 0.3408 | Train Acc: 0.9090
Epoch 5/5 | Loss: 0.2844 | Train Acc: 0.9213


model.eval()
→ Model ko evaluation/testing mode mein set karta hai.

correct = 0
→ Correct predictions ka count 0 se start karta hai.

with torch.no_grad():
→ Testing ke time gradient calculation band karta hai, memory aur computation save hoti hai.

for images, labels in test_loader:
→ Test data ko batch-by-batch leta hai.

images, labels = images.to(device), labels.to(device)
→ Images aur labels ko GPU/CPU par bhejta hai.

outputs = model(images)
→ Test images ko model mein pass karke predictions nikalta hai.

correct += (outputs.argmax(1) == labels).sum().item()
→ Correct predictions ko count karta hai.

test_acc = correct / len(test_dataset)
→ Total correct predictions se Test Accuracy calculate karta hai.

print(f"\nTest Accuracy: {test_acc:.4f}")
→ Final Test Accuracy print karta hai.

Overall Flow

Test Images → Model Prediction → Actual Labels se Compare → Correct Count → Test Accuracy

In [12]:
model.eval()
correct = 0
with torch.no_grad() :
  for images, labels in test_loader:
    images, labels = images.to(device), labels.to(device)
    outputs = model(images)
    correct += (outputs.argmax(1) == labels).sum().item()

test_acc = correct / len(test_dataset)
print(f"\nTest Accuracy : {test_acc:.4f}")


Test Accuracy : 0.9308


torch.save(model.state_dict(), "mnist_model.pth")
→ Trained model ke weights/parameters ko mnist_model.pth file mein save karta hai.

model.state_dict()
→ Model ke saare learned weights aur biases deta hai.

"mnist_model.pth"
→ Jis naam se model file save hogi.

print("Model saved as mnist_model.pth")
→ Confirm karta hai ki model successfully save ho gaya.

👉 Short mein:
Training ke baad model ke learned weights ko file mein save karta hai, taaki baad mein dobara training ki zarurat na pade.

In [13]:
torch.save(model.state_dict(), "mnist_model.pth")
print("Model saved as mnist_model.pth")

Model saved as mnist_model.pth
